# Phase 4 · CJPE · GQA-prenorm + LoRA (method control)
Control: identical architecture, settings and seed to the GQA LoRMA run (notebook 2); only the adaptation method changes (LoRA instead of LoRMA).
Needs `rr_lib.py` (v3) and `cjpe_lib.py` in `MyDrive/legal-llm-course/`. Input: a whole case, cut into {chunk}-token pieces, encoded by the frozen/adapted Phase 1 decoder, pooled by a learned head. Output: `submission.zip` with `prediction.json` = list of `{id, pred}` at the zip root (upload the file, not the folder).

In [ ]:
# 1 · Settings
from pathlib import Path
ACTION   = 'Train'          # Inspect | Train | Predict   (Predict = reload best.pt and rewrite the submissions)
PRESET   = 'GQA-prenorm'    # GQA-prenorm | MHA-prenorm | MHA-postnorm | MLA-prenorm
METHOD   = 'lora'           # full | partial | lora | rslora | lorma | ia3 | kd
RUN_ID   = 'gqa-lora-001'
SEED     = 17

# long-document representation (the experiment axes for the report)
CHUNK_LEN, STRIDE = 512, 512        # tokens per chunk; STRIDE < CHUNK_LEN = overlapping windows
MAX_CHUNKS = 8                      # chunks per case used in training/inference
SELECT   = 'last'      # first | last | uniform | head_tail | random   (which sections of the judgment)
AGG      = 'attn'            # mean | attn | bigru                         (how chunks are pooled)
MAX_TOKENS = 40000                  # keep head+tail of absurdly long cases when tokenising

# budget (T4 Colab): training uses DOCS_PER_EPOCH random cases per epoch; None = all
EPOCHS, DOCS_PER_EPOCH, MAX_MINUTES = 5, 2000, 60   # MAX_MINUTES: stop after the epoch that crosses it (best.pt is kept)
BATCH_DOCS, ACCUM, EVAL_DOCS = 4, 2, 8
CLASS_WEIGHT = 'sqrt'               # none | sqrt | inverse

COURSE_ROOT = Path('/content/drive/MyDrive/legal-llm-course')
PHASE_DIR   = COURSE_ROOT/'Phase_4_CJPE'
RUNS_ROOT   = COURSE_ROOT/'runs/phase4_cjpe'
BUNDLE_ZIP  = COURSE_ROOT/'phase2_lner_multimethod_bundle.zip'   # legal_scaffold, training_methods, tokenizer
RR_LIB      = COURSE_ROOT/'rr_lib.py'                            # Phase 3 helper (v3)
CJPE_LIB    = COURSE_ROOT/'cjpe_lib.py'                          # Phase 4 helper, upload next to rr_lib.py

# adapter / method knobs
RANK, ALPHA = 8, 16
LORA_TARGETS = ('query', 'value')
PARTIAL_LAYERS = (10, 11)
# knowledge distillation (METHOD='kd'): frozen teacher = a finished `full` run on the same chunks
KD_TEACHER_RUN = RUNS_ROOT/'MHA-prenorm/full/mha-full-001'
KD_STUDENT_LAYERS = (0, 2, 4, 6, 8, 10)                            # 12 -> 6 blocks copied from Phase 1
KD_TEMPERATURE, KD_ALPHA = 2.0, 0.3

DEFAULT_LR = {  # backbone / adapter / head
 'full': (3e-5, 0, 5e-4), 'kd': (3e-5, 0, 5e-4), 'partial': (1e-4, 0, 5e-4),
 'lora': (0, 1e-3, 5e-4), 'rslora': (0, 1e-3, 5e-4), 'lorma': (0, 1e-3, 5e-4), 'ia3': (0, 3e-3, 5e-4)}
assert ACTION in ('Inspect','Train','Predict') and METHOD in DEFAULT_LR and SELECT in ('first','last','uniform','head_tail','random')
PRESETS = {
 'GQA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='gqa',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MHA-postnorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=8,ffn_hidden=2048,context_length=2048,attention='mha',norm_placement='post',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02),
 'MLA-prenorm': dict(vocab_size=32768,d_model=512,layers=12,heads=8,kv_heads=2,ffn_hidden=2048,context_length=2048,attention='mla',norm_placement='pre',tie_embeddings=True,rope_base=10000,norm_epsilon=1e-6,init_std=.02,mla_latent=256,mla_query_rank=256,mla_rope_dim=32)}


In [ ]:
# 2 · Drive, helper bundle, tokenizer, device
from google.colab import drive
drive.mount('/content/drive')
import sys, shutil, zipfile, hashlib, json, subprocess, importlib.util, random
LOCAL = Path('/content/phase3-rr')
if LOCAL.exists(): shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE_ZIP) as z: z.extractall(LOCAL)
shutil.copy(RR_LIB, LOCAL/'rr_lib.py'); shutil.copy(CJPE_LIB, LOCAL/'cjpe_lib.py')
sys.path.insert(0, str(LOCAL))
need = [p for p in ('tokenizers','huggingface_hub','scikit-learn') if importlib.util.find_spec(p.replace('scikit-learn','sklearn')) is None]
if need: subprocess.check_call([sys.executable,'-m','pip','install','-q',*need])
import torch, rr_lib as R, cjpe_lib as J
from tokenizers import Tokenizer
from training_methods import resolve_snapshot_checkpoint
TOKENIZER_SHA256 = '906194848c1632431aac7d186e7c4f49da83cb6cbd480fa222c8472401cbf8ab'
tp = LOCAL/'tokenizer/tokenizer.json'
assert hashlib.sha256(tp.read_bytes()).hexdigest() == TOKENIZER_SHA256, 'tokenizer differs from Phase 1/2'
TOK = Tokenizer.from_file(str(tp))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); torch.manual_seed(SEED)
import numpy as np
print('device', device, torch.cuda.get_device_name(0) if device.type=='cuda' else '')

In [ ]:
# 3 · Data: tokenised cases come from a cache on Drive (built once with tools/build_token_cache.py, or by the first run)
CACHE = PHASE_DIR/f'tokens_v1_max{MAX_TOKENS}.npz'
DATA = J.load_cache(CACHE, TOKENIZER_SHA256, MAX_TOKENS)
if DATA is None:      # no usable cache: download (HF_TOKEN secret), tokenise, then save the cache for the next notebook
    from google.colab import userdata
    try: token = userdata.get('HF_TOKEN')
    except Exception: token = None
    J.ensure_files(COURSE_ROOT, token)
    DATA = J.load_all(PHASE_DIR)
    for s in DATA.values(): J.tokenize_docs(s, TOK, MAX_TOKENS)
    print('saved cache:', J.save_cache(DATA, CACHE, TOKENIZER_SHA256, MAX_TOKENS))
else:
    print('loaded tokenised cases from', CACHE)
print(json.dumps(J.summarize(DATA), indent=1))
ids = [d['id'] for d in DATA['test']]
assert len(ids) == len(set(ids)), 'duplicate test ids'
assert all(d['label'] in (0, 1) for d in DATA['train'] + DATA['dev']), 'labels must be 0/1'
assert all(d['label'] is None for d in DATA['test']), 'test labels must not be loaded'
print('chunks per case at CHUNK_LEN/STRIDE:', int(np.mean([len(J.chunk_spans(len(d["tok"]), CHUNK_LEN, STRIDE)) for d in DATA["train"]])), '(mean) | MAX_CHUNKS =', MAX_CHUNKS)

In [ ]:
# 4 · Build run config + model from the Phase 1 snapshot
lb, la, lh = DEFAULT_LR[METHOD]
C = dict(method=METHOD, preset=PRESET, run_id=RUN_ID, class_weight=CLASS_WEIGHT, epochs=EPOCHS, seed=SEED,
         lr=dict(backbone=lb, adapter=la, head=lh), partial_layers=PARTIAL_LAYERS,
         chunk_len=CHUNK_LEN, stride=STRIDE, max_chunks=MAX_CHUNKS, select=SELECT, agg=AGG,
         docs_per_epoch=DOCS_PER_EPOCH, max_minutes=MAX_MINUTES, batch_docs=BATCH_DOCS, accum=ACCUM, eval_docs=EVAL_DOCS,
         kd_temperature=KD_TEMPERATURE, kd_alpha=KD_ALPHA,
         head_kw=dict(hidden=256, agg=AGG, dropout=0.1),
         adapter_kw=dict(rank=RANK, alpha=ALPHA, targets=LORA_TARGETS),
         layer_keep=KD_STUDENT_LAYERS if METHOD=='kd' else None)
RUN_ROOT = RUNS_ROOT/PRESET/METHOD/RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

PHASE1_OVERRIDE = {'MHA-prenorm': str(COURSE_ROOT/'runs/mha_pre/run-001/legal-token-stages-v1/run-001-milestone-60762-aa4db459/state.pt'), 'MHA-postnorm': str(COURSE_ROOT/'runs/mha_post/run-001/legal-token-stages-v1/run-001-milestone-60762-53fcafd4/state.pt')}

def _scan_snapshots(preset):
    """Diagnostic: list every snapshot.json under runs/ (follows Drive shortcuts) vs the wanted config."""
    import os
    from legal_scaffold.models import ModelConfig
    want = ModelConfig(**PRESETS[preset]).validate().__dict__
    rows = []
    for dp, dn, fn in os.walk(COURSE_ROOT/'runs', followlinks=True):
        if 'snapshot.json' in fn:
            try:
                m = json.loads((Path(dp)/'snapshot.json').read_text())
                ident = m.get('identity', {}); cfg = ident.get('model', ident.get('config')) or {}
                got = ModelConfig(**cfg).validate().__dict__ if cfg else {}
                st = m.get('state_inventory', {}).get('model')
                rows.append((got == want, m.get('stage'), m.get('progress', {}).get('step'), bool(st and (Path(dp)/st).is_file()), dp))
            except Exception as e:
                rows.append((False, 'ERR '+str(e)[:40], None, False, dp))
    return rows

def phase1_checkpoint(preset):
    if preset in PHASE1_OVERRIDE:
        return Path(PHASE1_OVERRIDE[preset]), {'stage': 'override'}
    for kw in (dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=TOKENIZER_SHA256),
               dict(stages=('legal_continued','wikitext_only'), tokenizer_sha256=None),
               dict(stages=('legal_continued','legal_only','wikitext_only','legal','pretrain'), tokenizer_sha256=None)):
        try:
            return resolve_snapshot_checkpoint(COURSE_ROOT/'runs', PRESETS[preset], **kw)
        except FileNotFoundError:
            pass
    rows = _scan_snapshots(preset)
    print(f'--- No usable Phase 1 snapshot for {preset}. {len(rows)} snapshot.json files found under runs/ ---')
    for r in sorted(rows, key=lambda r: (not r[0], str(r[1])))[:25]:
        print('config_match=%s stage=%s step=%s payload_present=%s  %s' % r)
    raise FileNotFoundError(f'{preset}: no snapshot with matching config + payload; set PHASE1_OVERRIDE[{preset!r}] to its state.pt')



def build(C):
    ck, man = phase1_checkpoint(C['preset'])
    m = J.build_model(PRESETS[C['preset']], ck, 'full' if C['method']=='kd' else C['method'], head_kw=C['head_kw'],
                      adapter_kw=C['adapter_kw'] if C['method'] not in ('full','kd','partial') else {},
                      layer_keep=C['layer_keep'])
    return m, ck

if ACTION == 'Inspect':
    for p in PRESETS:
        try: print(p, '->', phase1_checkpoint(p)[0])
        except FileNotFoundError as e: print(p, '-> NOT FOUND')
    raise SystemExit('Inspect only.')
model, ckpt = build(C)
print('Phase 1 snapshot:', ckpt)
tr, tot = R.set_trainable(model, METHOD, C['partial_layers'])
print(f'trainable {tr:,} / {tot:,} ({100*tr/tot:.2f}%)   run folder: {RUN_ROOT}')

teacher = None
if METHOD == 'kd':
    tsaved = torch.load(KD_TEACHER_RUN/'best.pt', map_location='cpu', weights_only=False)
    tC = tsaved['C']
    assert tC['method'] == 'full', 'teacher should be a finished full fine-tune'
    for k in ('chunk_len','stride','max_chunks'): assert tC[k] == C[k], f'teacher/student must share {k}'
    teacher, _ = build(tC)
    teacher.load_state_dict(tsaved['trainable'], strict=False)
    teacher.to(device).eval()
    for p in teacher.parameters(): p.requires_grad_(False)
    n_t = sum(p.numel() for p in teacher.parameters()); n_s = sum(p.numel() for p in model.parameters())
    assert n_t > n_s, 'teacher must be larger than student'
    print(f'teacher {tC["preset"]} {n_t:,} params (dev macro-F1 {tsaved["dev"]["macro_f1"]:.4f}) -> student {PRESET}/{len(KD_STUDENT_LAYERS)} layers {n_s:,} params')

In [ ]:
# 5 · Train (resumes automatically from last.pt after a Colab disconnect)
if ACTION == 'Train':
    RESULTS, AMP = J.train_run(model, DATA, RUN_ROOT, C, device, teacher=teacher)
else:
    saved = torch.load(RUN_ROOT/'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(saved['trainable'], strict=False); model.to(device)
    AMP = torch.bfloat16 if (device.type=='cuda' and torch.cuda.is_bf16_supported()) else torch.float16
    RESULTS = json.loads((RUN_ROOT/'results.json').read_text())
d = RESULTS['dev']
print('best epoch', RESULTS['best_epoch'], '| dev macro-F1', round(d['macro_f1'],4), '| tuned', round(d['macro_f1_tuned'],4),
      f"@ {d['threshold_tuned']:.2f} | P {d['macro_precision']:.3f} R {d['macro_recall']:.3f} acc {d['accuracy']:.3f}")

In [ ]:
# 6 · Score dev + test, write BOTH submissions (threshold 0.5 and dev-tuned) and the probability files for ensembling
dev_res, P_DEV, P_TEST, OUT = J.export_run(model, DATA, C, device, AMP, RUN_ROOT)
print('upload one file, as-is:', {k: v['zip'] for k, v in OUT.items()})

In [ ]:
# 7 · Analysis for the report: classes, errors, sections, cost
import pandas as pd
dv = dev_res
print(pd.DataFrame(dv['per_class']).T.round(3).rename(index={'0':'label 0','1':'label 1'}))
print('confusion (rows = gold 0/1, cols = predicted 0/1):', dv['confusion'])
print('\nfalse positives / false negatives vs correct cases (document length in tokens):')
print(pd.DataFrame(J.error_table(DATA['dev'], P_DEV, 0.5)))
print('\nwhich part of the judgment matters? (this model, evaluated with different chunk selections, no retraining)')
abl = pd.DataFrame(J.selection_ablation(model, DATA['dev'], C, device, AMP)).pivot(index='select', columns='chunks', values='macro_f1')
display(abl)
if AGG in ('attn','bigru'):
    print('mean attention by position bin (start -> end of the selected chunks):', [round(x,3) for x in J.attention_profile(model, DATA['dev'], C, device, AMP)])
print(f"\ncost: {RESULTS['total_params']/1e6:.1f}M params, {RESULTS['trainable_params']/1e6:.2f}M trainable, "
      f"peak GPU {RESULTS['peak_gpu_gb']} GB, train {RESULTS['train_minutes']:.1f} min, inference {dv['infer_sec_per_doc']*1000:.0f} ms/case")
print(pd.DataFrame(RESULTS['history'])[['epoch','train_loss','dev_macro_f1','dev_macro_f1_tuned','minutes']].round(4))

In [ ]:
# 8 · Compare every finished run (for the report's tables)
import pandas as pd
rows = []
for f in sorted(RUNS_ROOT.rglob('results.json')):
    r = json.loads(f.read_text())
    rows.append({'run': f.parent.name, 'preset': r['preset'], 'method': r['method'], 'select': r['setup']['select'],
                 'agg': r['setup']['agg'], 'chunks': r['setup']['max_chunks'],
                 'trainable_M': round(r['trainable_params']/1e6,3), 'total_M': round(r['total_params']/1e6,1),
                 'best_ep': r['best_epoch'], 'dev_F1': round(r['dev']['macro_f1'],4), 'tuned_F1': round(r['dev']['macro_f1_tuned'],4),
                 'P': round(r['dev']['macro_precision'],4), 'R': round(r['dev']['macro_recall'],4),
                 'min': round(r['train_minutes'] or 0,1), 'peak_GB': r['peak_gpu_gb'] and round(r['peak_gpu_gb'],1),
                 'ms/case': round(r['dev']['infer_sec_per_doc']*1000)})
if rows: display(pd.DataFrame(rows).sort_values('dev_F1', ascending=False))